In [1]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from tqdm.auto import tqdm

In [ ]:
# =========================================================
# SETTINGS
# =========================================================

TARGET_H = 21
LAGS = (10, 21, 63)

ASSET_ORDER = [
    "US_EQUITY", "INTL_EQUITY", "US_BONDS", "INTL_BONDS",
    "US_REIT", "INTL_REIT", "GOLD", "BTC"
]

TRAIN_START = "2017-03-28"
TRAIN_END   = "2018-12-31"

VALID_START = "2019-01-01"
VALID_END   = "2020-12-31"

TEST_START  = "2021-01-01"
TEST_END    = "2025-12-31"

REFIT_EVERY = 1
RANDOM_STATE = 42
N_JOBS = -1

# RF hyperparameter grids
N_ESTIMATORS_GRID = [100, 200, 500]
MIN_SAMPLES_LEAF_GRID = [2, 5, 10, 20]
MAX_DEPTH_GRID = [10, 20, None]

# Rolling training window grid
WINDOW_GRID = [126, 189, 252, 320, 378, 441]

HAR_PATH = f"../proxy/realized_cov_target_h{TARGET_H}_lag_5_10_21_63.csv"

print("Target horizon :", TARGET_H)
print("Lags           :", LAGS)
print("HAR path       :", HAR_PATH)
print("Train period   :", TRAIN_START, "->", TRAIN_END)
print("Valid period   :", VALID_START, "->", VALID_END)
print("Test period    :", TEST_START, "->", TEST_END)

# =========================================================
# LOAD HAR MASTER FILE
# =========================================================
har = pd.read_csv(HAR_PATH, parse_dates=["Date"]).set_index("Date").sort_index()

print("HAR shape:", har.shape)
print("Date range:", har.index.min().date(), "->", har.index.max().date())
print()
print("First 10 columns:")
print(har.columns[:10].tolist())

# =========================================================
# BUILD RF DATASET FROM HAR MASTER FILE
# One row = one Date + one unique pair (i,j), i <= j
# =========================================================

def build_rf_dataset_from_har(har_df, asset_order, lags):
    rows = []

    unique_pairs = []
    for i, a in enumerate(asset_order):
        for j, b in enumerate(asset_order):
            if i <= j:
                unique_pairs.append((a, b))

    for date, row in har_df.iterrows():
        for a, b in unique_pairs:
            row_dict = {
                "Date": date,
                "pair": f"{a}__{b}",
                "target": row[f"target_cov_{a}__{b}"]
            }

            for lag in lags:
                row_dict[f"lag{lag}_cov"]   = row[f"lag{lag}_cov_{a}__{b}"]
                row_dict[f"lag{lag}_var_i"] = row[f"lag{lag}_cov_{a}__{a}"]
                row_dict[f"lag{lag}_var_j"] = row[f"lag{lag}_cov_{b}__{b}"]

            rows.append(row_dict)

    rf_df = pd.DataFrame(rows).sort_values(["Date", "pair"]).reset_index(drop=True)
    return rf_df


rf_df = build_rf_dataset_from_har(
    har_df=har,
    asset_order=ASSET_ORDER,
    lags=LAGS
)

print("RF dataset shape:", rf_df.shape)
print()
print(rf_df.head(10))

# =========================================================
# TRAIN / VALIDATION / TEST SPLIT
# =========================================================

train_mask = (rf_df["Date"] >= TRAIN_START) & (rf_df["Date"] <= TRAIN_END)
valid_mask = (rf_df["Date"] >= VALID_START) & (rf_df["Date"] <= VALID_END)
test_mask  = (rf_df["Date"] >= TEST_START)  & (rf_df["Date"] <= TEST_END)

rf_train = rf_df.loc[train_mask].copy()
rf_valid = rf_df.loc[valid_mask].copy()
rf_test  = rf_df.loc[test_mask].copy()

print("Train rows:", len(rf_train))
print("Validation rows:", len(rf_valid))
print("Test rows:", len(rf_test))
print()
print("Train dates:", rf_train["Date"].min().date(), "->", rf_train["Date"].max().date())
print("Valid dates:", rf_valid["Date"].min().date(), "->", rf_valid["Date"].max().date())
print("Test dates :", rf_test["Date"].min().date(),  "->", rf_test["Date"].max().date())


Target horizon : 21
Lags           : (10, 21, 63)
HAR path       : ../proxy/realized_cov_target_h21_lag_5_10_21_63.csv
Train period   : 2017-03-28 -> 2018-12-31
Valid period   : 2019-01-01 -> 2020-12-31
Test period    : 2021-01-01 -> 2025-12-31


HAR shape: (2121, 330)
Date range: 2017-06-27 -> 2025-12-02

First 10 columns:
['lag5_start', 'lag5_end', 'lag10_start', 'lag10_end', 'lag21_start', 'lag21_end', 'lag63_start', 'lag63_end', 'target_start', 'target_end']


In [29]:
# =========================================================
# TRAIN / VALIDATION / TEST SPLIT
# =========================================================

train_mask = (rf_df["Date"] >= TRAIN_START) & (rf_df["Date"] <= TRAIN_END)
valid_mask = (rf_df["Date"] >= VALID_START) & (rf_df["Date"] <= VALID_END)
test_mask  = (rf_df["Date"] >= TEST_START)  & (rf_df["Date"] <= TEST_END)

rf_train = rf_df.loc[train_mask].copy()
rf_valid = rf_df.loc[valid_mask].copy()
rf_test  = rf_df.loc[test_mask].copy()

print("Train rows:", len(rf_train))
print("Validation rows:", len(rf_valid))
print("Test rows:", len(rf_test))
print()
print("Train dates:", rf_train["Date"].min().date(), "->", rf_train["Date"].max().date())
print("Valid dates:", rf_valid["Date"].min().date(), "->", rf_valid["Date"].max().date())
print("Test dates :", rf_test["Date"].min().date(),  "->", rf_test["Date"].max().date())

# =========================================================
# FEATURE MATRIX
# =========================================================

FEATURE_COLS = [
    "lag10_cov", "lag21_cov", "lag63_cov",
    "lag10_var_i", "lag10_var_j",
    "lag21_var_i", "lag21_var_j",
    "lag63_var_i", "lag63_var_j"
]

TARGET_COL = "target"

X_train = rf_train[FEATURE_COLS].values
y_train = rf_train[TARGET_COL].values

X_valid = rf_valid[FEATURE_COLS].values
y_valid = rf_valid[TARGET_COL].values

print()
print("Train X shape:", X_train.shape)
print("Train y shape:", y_train.shape)
print("Valid X shape:", X_valid.shape)
print("Valid y shape:", y_valid.shape)

# =========================================================
# MATRIX RECONSTRUCTION
# =========================================================

def build_cov_matrix(df, value_col, asset_order):
    n = len(asset_order)
    mat = np.zeros((n, n))

    for _, row in df.iterrows():
        a, b = row["pair"].split("__")
        i = asset_order.index(a)
        j = asset_order.index(b)

        mat[i, j] = row[value_col]
        mat[j, i] = row[value_col]

    return mat

# =========================================================
# METRICS
# =========================================================

def matrix_qlike(S_true, H_pred, ridge_eps=1e-10):
    n = S_true.shape[0]
    I = np.eye(n)

    S = 0.5 * (S_true + S_true.T) + ridge_eps * I
    H = 0.5 * (H_pred + H_pred.T) + ridge_eps * I

    try:
        H_inv = np.linalg.inv(H)
        A = S @ H_inv
        sign, logdet = np.linalg.slogdet(A)

        if sign <= 0:
            return np.nan

        return float(np.trace(A) - logdet - n)

    except np.linalg.LinAlgError:
        return np.nan


def frobenius_distance(S_true, H_pred):
    S = 0.5 * (S_true + S_true.T)
    H = 0.5 * (H_pred + H_pred.T)
    return float(np.linalg.norm(S - H, ord="fro"))

# =========================================================
# STABLE PD REPAIR
# =========================================================

def make_pd_stable(M, start_alpha=1e-8, max_alpha=1.0, growth=10.0, return_alpha=False):
    M = 0.5 * (M + M.T)
    I = np.eye(M.shape[0])

    scale = np.mean(np.diag(M))
    if not np.isfinite(scale) or scale <= 0:
        scale = 1.0

    alpha = start_alpha * scale

    while alpha <= max_alpha * scale:
        M_try = M + alpha * I
        try:
            np.linalg.cholesky(M_try)
            if return_alpha:
                return M_try, alpha
            return M_try
        except np.linalg.LinAlgError:
            alpha *= growth

    eigvals, eigvecs = np.linalg.eigh(M)
    floor = 1e-4 * scale
    eigvals = np.clip(eigvals, floor, None)
    M_psd = eigvecs @ np.diag(eigvals) @ eigvecs.T
    M_psd = 0.5 * (M_psd + M_psd.T)

    if return_alpha:
        return M_psd, np.nan
    return M_psd

# =========================================================
# EVALUATE ONE RF SPECIFICATION
# =========================================================

def evaluate_rf_spec(
    rf_df,
    har,
    feature_cols,
    target_col,
    asset_order,
    target_h,
    validation_start,
    validation_end,
    training_window,
    n_estimators,
    min_samples_leaf,
    max_depth,
    refit_every=5,
    random_state=42,
    n_jobs=-1,
    verbose=True
):
    validation_start = pd.Timestamp(validation_start)
    validation_end = pd.Timestamp(validation_end)

    validation_dates = har.index[
        (har.index >= validation_start) &
        (har.index <= validation_end)
    ]

    qlike_list = []
    frob_list = []
    raw_min_eigs = []
    repair_alphas = []
    qlike_by_date = []

    current_model = None
    last_refit_loc = None
    n_refits = 0

    for d in validation_dates:
        d_loc = har.index.get_loc(d)

        should_refit = (
            current_model is None
            or last_refit_loc is None
            or (d_loc - last_refit_loc) >= refit_every
        )

        if should_refit:
            train_end_loc = d_loc - target_h
            train_start_loc = train_end_loc - training_window + 1

            if train_start_loc < 0:
                continue

            train_start_date = har.index[train_start_loc]
            train_end_date = har.index[train_end_loc]

            train_slice = rf_df[
                (rf_df["Date"] >= train_start_date) &
                (rf_df["Date"] <= train_end_date)
            ].copy()

            if len(train_slice) == 0:
                continue

            X_train_roll = train_slice[feature_cols].values
            y_train_roll = train_slice[target_col].values

            current_model = RandomForestRegressor(
                n_estimators=n_estimators,
                min_samples_leaf=min_samples_leaf,
                max_depth=max_depth,
                random_state=random_state,
                n_jobs=n_jobs
            )

            current_model.fit(X_train_roll, y_train_roll)
            last_refit_loc = d_loc
            n_refits += 1

        pred_slice = rf_df[rf_df["Date"] == d].copy()

        if len(pred_slice) == 0 or current_model is None:
            continue

        X_pred = pred_slice[feature_cols].values
        pred_slice["pred"] = current_model.predict(X_pred)

        H_pred = build_cov_matrix(pred_slice, "pred", asset_order)
        S_true = build_cov_matrix(pred_slice, target_col, asset_order)

        raw_min_eigs.append(np.min(np.linalg.eigvalsh(H_pred)))

        H_pred_pd, alpha_used = make_pd_stable(H_pred, return_alpha=True)
        repair_alphas.append(alpha_used)

        q = matrix_qlike(S_true, H_pred_pd)
        f = frobenius_distance(S_true, H_pred_pd)

        if np.isfinite(q):
            qlike_list.append(q)
            qlike_by_date.append((d, q))

        if np.isfinite(f):
            frob_list.append(f)

    qlike_series = pd.Series(qlike_list, dtype=float)
    frob_series = pd.Series(frob_list, dtype=float)

    top_bad_dates = sorted(qlike_by_date, key=lambda x: x[1], reverse=True)[:5]

    result = {
        "training_window": training_window,
        "n_estimators": n_estimators,
        "min_samples_leaf": min_samples_leaf,
        "max_depth": max_depth,
        "refit_every": refit_every,
        "QLIKE_mean": qlike_series.mean() if len(qlike_series) > 0 else np.nan,
        "QLIKE_median": qlike_series.median() if len(qlike_series) > 0 else np.nan,
        "QLIKE_p95": qlike_series.quantile(0.95) if len(qlike_series) > 0 else np.nan,
        "Frobenius_mean": frob_series.mean() if len(frob_series) > 0 else np.nan,
        "avg_raw_min_eig": np.mean(raw_min_eigs) if len(raw_min_eigs) > 0 else np.nan,
        "worst_raw_min_eig": np.min(raw_min_eigs) if len(raw_min_eigs) > 0 else np.nan,
        "avg_alpha_used": np.nanmean(repair_alphas) if len(repair_alphas) > 0 else np.nan,
        "max_alpha_used": np.nanmax(repair_alphas) if len(repair_alphas) > 0 else np.nan,
        "n_obs": len(qlike_series),
        "n_refits": n_refits,
        "top_bad_dates": top_bad_dates
    }

    if verbose:
        print(
            f"window={training_window:>3} | "
            f"trees={n_estimators:>3} | "
            f"leaf={min_samples_leaf:>2} | "
            f"depth={str(max_depth):>4} | "
            f"QLIKE_mean={result['QLIKE_mean']:.3f} | "
            f"QLIKE_med={result['QLIKE_median']:.3f} | "
            f"Frob={result['Frobenius_mean']:.6f} | "
            f"obs={result['n_obs']} | "
            f"refits={result['n_refits']}"
        )

    return result
# =========================================================
# QUICK SANITY TESTS
# =========================================================

res = evaluate_rf_spec(
    rf_df=rf_df,
    har=har,
    feature_cols=FEATURE_COLS,
    target_col=TARGET_COL,
    asset_order=ASSET_ORDER,
    target_h=TARGET_H,
    validation_start=VALID_START,
    validation_end=VALID_END,
    training_window=252,
    n_estimators=200,
    min_samples_leaf=5,
    max_depth=None,
    refit_every=5,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    verbose=True
)

res

Train rows: 13716
Validation rows: 18180
Test rows: 44460

Train dates: 2017-06-27 -> 2018-12-31
Valid dates: 2019-01-02 -> 2020-12-31
Test dates : 2021-01-04 -> 2025-12-02

Train X shape: (13716, 9)
Train y shape: (13716,)
Valid X shape: (18180, 9)
Valid y shape: (18180,)
window=252 | trees=200 | leaf= 5 | depth=None | QLIKE_mean=3272.478 | QLIKE_med=20.614 | Frob=0.004211 | obs=505 | refits=101


{'training_window': 252,
 'n_estimators': 200,
 'min_samples_leaf': 5,
 'max_depth': None,
 'refit_every': 5,
 'QLIKE_mean': np.float64(3272.478106982936),
 'QLIKE_median': np.float64(20.61439445615268),
 'QLIKE_p95': np.float64(17949.85637884396),
 'Frobenius_mean': np.float64(0.004211381529121081),
 'avg_raw_min_eig': np.float64(-0.0005437935046023725),
 'worst_raw_min_eig': np.float64(-0.008890134716235927),
 'avg_alpha_used': np.float64(0.00022603360621580474),
 'max_alpha_used': np.float64(0.0033405137497708346),
 'n_obs': 505,
 'n_refits': 101,
 'top_bad_dates': [(Timestamp('2020-03-12 00:00:00'), 107020.49407536672),
  (Timestamp('2020-03-10 00:00:00'), 90977.95982247336),
  (Timestamp('2020-07-10 00:00:00'), 55166.134405157696),
  (Timestamp('2020-10-21 00:00:00'), 54683.04239804541),
  (Timestamp('2020-10-20 00:00:00'), 47970.87406139914)]}